# CISC440 – Homework 9  
# Misère Nim: From Search to Learning  
## Monte Carlo Simulation and Reinforcement Learning

Name: Nikhil Jangir and Rudy Vergara

## Overview

In Homework 6, you solved **Misère Nim** using adversarial search methods such as Minimax, Alpha-Beta Pruning, and Expectimax. Those methods assume the agent reasons using explicit search trees.

In this homework, you will revisit the same game but move into a new area of AI:

- **Monte Carlo methods**: learning through simulation
- **Reinforcement Learning**: learning through repeated experience

Instead of solving the game by searching the tree exactly, your agent will estimate strong moves through simulation and learn policies through trial and error.

This homework continues the same Misère Nim environment so you can focus on new AI ideas rather than learning a new game.

## Learning Objectives

By completing this homework, you will be able to:

- Apply Monte Carlo simulation to estimate action quality
- Use repeated rollouts to choose moves
- Model learning agents using states, actions, and rewards
- Implement Q-learning for sequential decision making
- Compare search-based and learning-based approaches
- Analyze experimental performance of AI agents

## Misère Nim Rules

You are given several heaps of objects.

A move consists of:

1. Selecting one heap
2. Removing one or more objects from that heap

The player who removes the **last remaining object loses**.

This is different from normal Nim, where the player who removes the last object usually wins.

## Initial States for Testing

Use the following starting states:

```python
[1, 3, 5]
[3, 5, 7]
[1, 1, 2]
[2, 4, 6]
```

## Submission Requirements

Submit:

1. This completed Jupyter notebook
2. Any helper Python files you created
3. Clear outputs for all required experiments
4. Written explanations in the markdown cells
5. Clear comments in code

You may reuse parts of your Homework 6 Nim code, but your Monte Carlo and Q-learning work must be your own.

## Grading Breakdown: 100 Points

| Section | Points |
|---|---:|
| Layer 1 – Problem Formulation | 25 |
| Layer 2 – Monte Carlo Implementation | 30 |
| Layer 3 – Reinforcement Learning | 30 |
| Layer 4 – Experimental Comparison | 15 |

## Setup Code

In [26]:
import random
from collections import defaultdict
import pandas as pd

# Helper Functions

You may complete or modify these helper functions. These are intentionally simple so that the focus stays on Monte Carlo simulation and reinforcement learning.

In [27]:
def normalize_state(state):
    """
    Convert a state to a tuple so it can be used as a dictionary key.
    Example:
        [3, 5, 7] -> (3, 5, 7)
    """
    return tuple(state)


def is_terminal(state):
    """
    Return True if the game is over.
    In Nim, the game ends when all heaps are empty.
    """
    # The game is terminal when the sum of all heap sizes is 0
    return sum(state) == 0


def legal_actions(state):
    """
    Return all legal actions from a state.

    Action format:
        (heap_index, remove_count)
    """
    actions = []
    for heap_idx, heap_size in enumerate(state):
        # For each non-empty heap, we can remove 1 to heap_size objects
        for remove_count in range(1, heap_size + 1):
            actions.append((heap_idx, remove_count))
    return actions


def apply_action(state, action):
    """
    Apply an action and return the new state.
    Do not modify the original state.
    """
    heap_idx, remove_count = action
    new_state = list(state)  # Create a copy to avoid modifying the original
    new_state[heap_idx] -= remove_count  # Remove objects from the heap
    return new_state

# Layer 1 – Problem Formulation (25 Points)

Answer clearly in complete sentences.

## Part A – Monte Carlo Modeling (12 pts)

### Q1. State Representation (3 pts)

How will a Misère Nim board state be represented in Python?

Example:

```python
[3, 5, 7]
```

Explain what each number means.

**Your answer:**

A Misère Nim board state is represented as a list of integers, where each integer represents the number of objects in a single heap. Refrencing the example above: the first heap contains 3 objects, the second heap contains 5 objects, and the third heap contains 7 objects. The index position in the list identifies which heap it is (heap 0, heap 1, heap 2). 

### Q2. Legal Actions (3 pts)

Describe all legal actions available from:

```python
[1, 3, 5]
```

Represent actions as:

```python
(heap_index, remove_count)
```

**Your answer:**

From state `[1, 3, 5]`, the legal actions are:
- From heap 0: `(0, 1)` = remove 1 object
- From heap 1: `(1, 1)`, `(1, 2)`, `(1, 3)` = remove 1, 2, or 3 objects
- From heap 2: `(2, 1)`, `(2, 2)`, `(2, 3)`, `(2, 4)`, `(2, 5)` = remove 1, 2, 3, 4, or 5 objects

In total, there are 9 legal actions. A legal action consists of selecting one non-empty heap and removing at least one object (but not more than the heap size). Each action is represented as a tuple `(heap_index, remove_count)` where `heap_index` identifies which heap to remove from, and `remove_count` specifies how many objects to remove.

### Q3. Terminal Test (3 pts)

When does the game end?

How does Misère Nim differ from normal Nim?

**Your answer:**

In Misère Nim, the game ends when all heaps are empty which is when the state is `[0, 0, 0]`. At this point, there are no legal moves remaining.

The critical difference is in normal Nim, the player who takes the last object wins. This reversal fundamentally changes game strategy. In Misère Nim, players want to avoid being forced into a position where they must take the last remaining object. This makes certain positions that would be winning in normal Nim actually losing positions in Misère Nim, and vice versa.

### Q4. Why Monte Carlo? (3 pts)

Why might simulation be useful instead of exploring the full search tree?

**Your answer:**

Monte Carlo simulation is useful for several reasons:

1. The full game tree can be exponentially large. Sampling trajectories is often faster than exhaustively exploring all branches.

2. For many games, you don't need the exact game value; a good estimate based on random rollouts is often good enough to make strong moves.

3. As game complexity increases (more heaps, larger heap sizes), the search tree becomes intractable.

4. No need to compute minimax values or maintain complex game state trees in memory. Just simulate random games and count wins/losses.


### Q5. Define State (3 pts)

What is the RL state in this problem?

**Your answer:**

The RL state is the current configuration of heaps on the board, represented as a list of integers (e.g., `[1, 3, 5]`). The state captures all the information needed to decide what action to take next.

### Q6. Define Actions (3 pts)

What are RL actions in this problem?

**Your answer:**

RL actions are the same as legal moves in Misère Nim: tuples of the form `(heap_index, remove_count)` representing which heap to remove from and how many objects to remove. From any state, we generate all legal actions using the `legal_actions()` function. Each action transitions the current state to a new state via the `apply_action()` function.

### Q7. Reward Function (4 pts)

Define rewards for:

- Winning
- Losing
- Non-terminal moves

Explain your choice.

**Your answer:**

**Reward structure:**
- **Winning** (opponent reaches terminal state with objects left): `+1`
- **Losing** (current player forced to take last object): `-1`
- **Non-terminal moves**: `0`

**Explanation:** This simple reward structure aligns perfectly with the game objective. We want the agent to maximize rewards, so winning gives +1 and losing gives -1. Intermediate moves get 0 reward;

### Q8. Exploration vs Exploitation (3 pts)

What is the difference between exploration and exploitation?

Why are both important?

**Your answer:**

**Exploration vs Exploitation:**
- **Exploitation**: Choosing the action with the highest known Q-value. This uses what we've already learned to play well.
- **Exploration**: Occasionally choosing a random action instead. This lets us try new moves and discover better strategies we haven't found yet.

**Why both are important:**
- Both are imortant because Iif we only exploit early estimates, we may miss better strategies or get stuck at local optima. Whereas, exploration alone means we never leverage what we've learned; we'd play randomly forever and not improve. That's why balancing both of them is the key.

# Layer 2 – Monte Carlo Implementation (30 Points)

Implement a simulation-based agent.

## Part A – Random Playout Engine (10 pts)

Write:

```python
simulate_random_game(state)
```

This function should:

- Start from the given state
- Alternate players randomly
- Choose random legal moves
- Return the winner

For this homework, use player `0` as the starting player and player `1` as the other player.

Remember: in Misère Nim, the player who takes the last object loses.

In [28]:
def simulate_random_game(state, starting_player=0):
    """
    Simulate a full random Misère Nim game.

    Parameters:
        state: list of heap sizes
        starting_player: 0 or 1

    Returns:
        winner: 0 or 1
    """
    current_state = list(state)
    current_player = starting_player
    
    # Play until game is terminal (all heaps are empty)
    while not is_terminal(current_state):
        # Get all legal actions from current state
        actions = legal_actions(current_state)
        
        # Choose a random action
        action = random.choice(actions)
        
        # Apply the action to get new state
        current_state = apply_action(current_state, action)
        
        # Switch to other player
        current_player = 1 - current_player
    
    winner = current_player
    return winner

### Test your random playout engine

In [29]:
# Test state functions
print("=== Testing State Functions ===\n")

# Test is_terminal
print("Testing is_terminal():")
print(f"  is_terminal([0, 0, 0]): {is_terminal([0, 0, 0])} (should be True)")
print(f"  is_terminal([1, 3, 5]): {is_terminal([1, 3, 5])} (should be False)")
print(f"  is_terminal([0, 0, 1]): {is_terminal([0, 0, 1])} (should be False)\n")

# Test legal_actions
print("Testing legal_actions():")
state = [1, 3, 5]
actions = legal_actions(state)
print(f"  legal_actions({state}):")
print(f"  Total actions: {len(actions)}")
print(f"  Actions: {actions}\n")

# Test apply_action
print("Testing apply_action():")
state = [1, 3, 5]
action = (1, 2)  # Remove 2 from heap 1
new_state = apply_action(state, action)
print(f"  Original state: {state}")
print(f"  Action: {action} (remove {action[1]} from heap {action[0]})")
print(f"  New state: {new_state}")
print(f"  Original state unchanged: {state}\n")  # Verify original unchanged

# Test simulate_random_game
print("=== Testing Monte Carlo Functions ===\n")
print("Testing simulate_random_game():")
for i in range(5):
    winner = simulate_random_game([1, 3, 5])
    print(f"  Game {i+1}: Player {winner} wins")
print()

# Test rollout_value
print("Testing rollout_value():")
state = [1, 3, 5]
action = (2, 1)  # Remove 1 from heap 2
value = rollout_value(state, action, n_trials=50)
print(f"  State: {state}")
print(f"  Action: {action}")
print(f"  Estimated win rate: {value}\n")


=== Testing State Functions ===

Testing is_terminal():
  is_terminal([0, 0, 0]): True (should be True)
  is_terminal([1, 3, 5]): False (should be False)
  is_terminal([0, 0, 1]): False (should be False)

Testing legal_actions():
  legal_actions([1, 3, 5]):
  Total actions: 9
  Actions: [(0, 1), (1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5)]

Testing apply_action():
  Original state: [1, 3, 5]
  Action: (1, 2) (remove 2 from heap 1)
  New state: [1, 1, 5]
  Original state unchanged: [1, 3, 5]

=== Testing Monte Carlo Functions ===

Testing simulate_random_game():
  Game 1: Player 1 wins
  Game 2: Player 1 wins
  Game 3: Player 0 wins
  Game 4: Player 0 wins
  Game 5: Player 1 wins

Testing rollout_value():
  State: [1, 3, 5]
  Action: (2, 1)
  Estimated win rate: 0.56



## Part B – Action Evaluation by Rollouts (10 pts)

Write:

```python
rollout_value(state, action, n_trials=100)
```

This function should:

1. Apply the chosen action
2. Simulate random completions
3. Estimate the win rate for the player who made the original action

In [30]:
def rollout_value(state, action, n_trials=100):
    """
    Estimate the value of an action using random rollouts.

    Return:
        estimated win rate for the current player
    """
    # Apply the action to transition to next state
    next_state = apply_action(state, action)
    next_player = 1
    
    wins = 0
    # Run multiple random games from this state
    for _ in range(n_trials):
        # Simulate a random game from the new state
        winner = simulate_random_game(next_state, next_player)
        
        # Current player (player 0, who made this action) wins if winner is NOT player 1
        if winner == 0:
            wins += 1
    
    return wins / n_trials

## Part C – Monte Carlo Move Selection (10 pts)

Write:

```python
best_move_mc(state, n_trials=100)
```

Return the action with the highest estimated win probability.

In [31]:
def best_move_mc(state, n_trials=100):
    """
    Choose the best move using Monte Carlo rollouts.

    Return:
        best_action, results_table

    results_table can be a list of dictionaries or a pandas DataFrame.
    """
    actions = legal_actions(state)
    results = []
    
    best_action = None
    best_value = -1
    
    # Evaluate each legal action
    for action in actions:
        value = rollout_value(state, action, n_trials=n_trials)
        
        # Store results for reporting
        results.append({
            'action': action,
            'win_rate': value
        })
        
        # Track the best action
        if value > best_value:
            best_value = value
            best_action = action
    
    df = pd.DataFrame(results)
    df = df.sort_values('win_rate', ascending=False)  # Sort by win rate descending
    
    return best_action, df

## Required Monte Carlo Testing

Run your Monte Carlo agent on:

```python
[1, 3, 5]
[3, 5, 7]
[1, 1, 2]
```

For each state report:

- Move tested
- Estimated win rate
- Best move selected

Use a clear table.

In [32]:
test_states = [
    [1, 3, 5],
    [3, 5, 7],
    [1, 1, 2]
]

# Run best_move_mc for each state and display results
print("=== Monte Carlo Results ===\n")

for state in test_states:
    print(f"State: {state}")
    best_action, results_df = best_move_mc(state, n_trials=100)
    print(f"Best action: {best_action} with win rate: {results_df.iloc[0]['win_rate']}\n")
    print("All actions evaluated:")
    print(results_df.to_string(index=False))
    print("\n" + "="*50 + "\n")

=== Monte Carlo Results ===

State: [1, 3, 5]
Best action: (1, 1) with win rate: 0.62

All actions evaluated:
action  win_rate
(1, 1)      0.62
(2, 2)      0.57
(2, 5)      0.56
(1, 3)      0.54
(2, 1)      0.53
(0, 1)      0.51
(2, 3)      0.50
(1, 2)      0.48
(2, 4)      0.48


State: [3, 5, 7]
Best action: (0, 2) with win rate: 0.62

All actions evaluated:
action  win_rate
(0, 2)      0.62
(2, 6)      0.55
(2, 7)      0.55
(1, 2)      0.53
(1, 1)      0.50
(2, 3)      0.50
(1, 4)      0.49
(2, 4)      0.49
(1, 5)      0.48
(2, 1)      0.48
(0, 1)      0.47
(0, 3)      0.45
(2, 2)      0.45
(1, 3)      0.44
(2, 5)      0.43


State: [1, 1, 2]
Best action: (2, 1) with win rate: 1.0

All actions evaluated:
action  win_rate
(2, 1)      1.00
(0, 1)      0.55
(1, 1)      0.51
(2, 2)      0.00




# Layer 3 – Reinforcement Learning (30 Points)

Implement a Q-learning agent for Misère Nim.

## Part A – Q Table Design (5 pts)

How will you store:

```python
Q[state][action]
```

Explain your structure.

**Your answer:**

## Part B – Q-learning Update Rule (10 pts)

Use:

```text
Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') − Q(s,a) ]
```

Implement:

```python
update_q(Q, s, a, r, s_next)
```

In [33]:
def get_q(Q, state, action):
    """
    Safely get Q-value for a state-action pair.
    """
    return Q[normalize_state(state)][action]


def update_q(Q, state, action, reward, next_state, alpha=0.5, gamma=0.9):
    """
    Apply the Q-learning update.
    """
    # TODO: implement Q-learning update
    pass

## Part C – Epsilon-Greedy Action Selection

Implement an epsilon-greedy policy.

- With probability epsilon: choose a random legal action
- Otherwise: choose the action with the highest Q-value

In [34]:
def choose_action_epsilon_greedy(Q, state, epsilon=0.1):
    """
    Choose an action using epsilon-greedy exploration.
    """
    # TODO: implement epsilon-greedy action selection
    pass

## Part D – Training by Self-Play (10 pts)

Train your agent by self-play for at least:

```python
3000 episodes
```

You may train one shared Q-table for both players.

In [35]:
def train_q_learning(n_episodes=3000, start_states=None, alpha=0.5, gamma=0.9, epsilon=0.2):
    """
    Train a Q-learning agent through self-play.

    Returns:
        Q-table
        training statistics
    """
    if start_states is None:
        start_states = [[1, 3, 5], [3, 5, 7], [1, 1, 2], [2, 4, 6]]

    Q = defaultdict(lambda: defaultdict(float))

    # TODO: implement training loop

    return Q

## Part E – Learned Policy (5 pts)

After training, report the best learned move for:

```python
[1, 3, 5]
[3, 5, 7]
[1, 1, 2]
```

In [36]:
# TODO: train Q-learning agent and report best learned moves
# Q = train_q_learning(n_episodes=3000)

# Layer 4 – Experimental Comparison (15 Points)

Run tournaments of 20 games each.

## Required Matchups

1. Random vs Random
2. Monte Carlo vs Random
3. Q-learning vs Random
4. Q-learning vs Monte Carlo
5. Minimax from Homework 6 vs Q-learning

If you do not reuse your Minimax agent from Homework 6, clearly state that and compare the first four matchups.

In [37]:
def random_agent(state):
    """
    Choose a random legal action.
    """
    return random.choice(legal_actions(state))


def mc_agent(state):
    """
    Choose an action using Monte Carlo rollouts.
    """
    action, _ = best_move_mc(state, n_trials=100)
    return action


def q_learning_agent(Q, state):
    """
    Choose the best action according to learned Q-values.
    """
    actions = legal_actions(state)
    if not actions:
        return None
    return max(actions, key=lambda a: Q[normalize_state(state)][a])


def play_game(agent0, agent1, start_state):
    """
    Play one game between two agents.

    agent0 and agent1 are functions that take state and return action.

    Return:
        winner: 0 or 1
    """
    # TODO: implement game playing between two agents
    pass


def run_tournament(agent0, agent1, start_state=[3, 5, 7], n_games=20):
    """
    Run a tournament and return win counts.
    """
    # TODO: implement tournament
    pass

## Report Table

Fill in a table like this:

| Matchup | Agent 1 Wins | Agent 2 Wins | Winner |
|---|---:|---:|---|
| Random vs Random | | | |
| Monte Carlo vs Random | | | |
| Q-learning vs Random | | | |
| Q-learning vs Monte Carlo | | | |
| Minimax vs Q-learning | | | |

In [38]:
# TODO: run tournaments and display results

## Analysis Questions

### Q1 (5 pts)

Which agent performed best overall?

**Your answer:**

### Q2 (5 pts)

Which method required the most computation?

**Your answer:**

### Q3 (5 pts)

When is search better than learning?

When is learning better than search?

**Your answer:**

# Code Quality Expectations

Your code should include:

- Meaningful variable names
- Clear functions
- Comments where needed
- Organized outputs
- Tables or printed summaries for experiments

# Summary

Modify rewards to encourage shorter wins or longer survival.

Did the learned policy change?

Explain.

**Your answer:**

# Hints

- Reuse Homework 6 Nim code if helpful
- Normalize states as tuples for dictionary keys
- Use the `random` module carefully
- Begin with small states before `[3, 5, 7]`
- Test helper functions before building agents

# Academic Integrity

You may discuss ideas, but all code and written work must be your own.

You may not submit another student’s implementation or an AI-generated solution without understanding and modifying it yourself.

# Final Reflection

This homework shows a major transition in AI:

- Search computes strong moves
- Simulation estimates strong moves
- Reinforcement learning discovers strong moves

That is one of the most important ideas in modern AI.